In [1]:
import sys
sys.path.append('../../')
from utilities.data import data
data.BASEPATH = '../../data/'
from utilities.utils import utils
activities = data.extractAirBnbData()
utils.createTopicsFromText(activities,"description")
act_matrix = utils.createDataMatrix(activities, ["rating", "numberOfReviews","numofPics","price"])

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Shahzad\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\Shahzad\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\Shahzad\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
2023-12-25 20:06:29,087 - top2vec - INFO - Pre-processing documents for training
2023-12-25 20:06:29,687 - top2vec - INFO - Creating joint document/word embedding
2023-12-25 20:06:36,695 - top2vec - INFO - Creating lower dimension embedding of documents
2023-12-25 20:06:44,832 - top2vec - INFO - Finding dense areas of documents
2023-12-25 20:06:44,880 - top2vec - INFO - Finding topics


In [2]:
users = data.getAirBnbUsersData(activities)
user_nodes = utils.generateUserData(users)
user_matrix = utils.createDataMatrix(user_nodes, ["numberOfReview", "average_rating"])
links, link_features = utils.generateUserLinks(users,activities)

In [3]:
import numpy as np
location_map = utils.generateTopicEmbeddings(activities,'location')
location_embedding = utils.createEmbeddingMatrix(location_map,activities,"location")
description_map = utils.generateTopicEmbeddings(activities,'description_topics')
description_embedding = utils.createEmbeddingMatrix(description_map,activities,"description_topics")
act_matrix1 = np.hstack((act_matrix,location_embedding,description_embedding))

In [4]:
from torch.utils.data import random_split
import torch
from torch_geometric.data import Data
def getByindices(links, indices,getOnlyRatingAndSentiment=False):
    result = []
    for i in indices:
        if(getOnlyRatingAndSentiment):
            result.append(links[i][0:4])
        else:
            result.append(links[i])          
    return result

# Features for each node (for simplicity, random features are used here)
left_features = torch.from_numpy(user_matrix).float()
right_features = torch.from_numpy(act_matrix1).float()

train_size = int(0.9 * len(links))
val_size = int(0.05 * len(links))
test_size = len(links) - train_size - val_size

train_link, val_link, test_link = random_split(links, [train_size, val_size, test_size])

train_edges = torch.tensor(getByindices(links,train_link.indices), dtype=torch.long).t()
validate_edges = torch.tensor(getByindices(links,val_link.indices), dtype=torch.long).t()
test_edges = torch.tensor(getByindices(links,test_link.indices), dtype=torch.long).t()


train_edge_features_full = torch.from_numpy(np.array(getByindices(link_features,train_link.indices))).float()
train_edge_features = torch.from_numpy(np.array(getByindices(link_features,train_link.indices,True))).float()

validate_edge_features = torch.from_numpy(np.array(getByindices(link_features,val_link.indices,True))).float()
test_edge_features = torch.from_numpy(np.array(getByindices(link_features,test_link.indices,True))).float()

data = Data(x_left=left_features,x_right=right_features, edge_index=train_edges, edge_attr = train_edge_features_full )

In [ ]:
from torch_geometric.nn import GCNConv,GATConv,GraphConv, LayerNorm,PDNConv

class LinkPredictor(torch.nn.Module):
    def __init__(self, num_left_node_features,num_right_node_features,edge_features):
        super(LinkPredictor, self).__init__()
        self.left_embedding = torch.nn.Linear(num_left_node_features, 128)
        self.right_embedding = torch.nn.Linear(num_right_node_features, 128)
        self.norm0 = LayerNorm(128)
        self.conv1 = GATConv(128, 256)
        self.norm1 = LayerNorm(256)
        self.conv2 = PDNConv(256, 350,edge_features,edge_features)
        self.norm2 = LayerNorm(350)
        self.conv3 = GCNConv(350, 250)
        self.norm3 = LayerNorm(250)
        self.link_features = torch.nn.Linear(2*250, 160)
        self.norm4 = LayerNorm(160)
        self.link_features2 = torch.nn.Linear(160, 128)
        self.norm5 = LayerNorm(128)
        self.link_features3 = torch.nn.Linear(128, 4)
        self.dropout = torch.nn.Dropout(0.2)


    def forward(self, data):
        x_left,x_right, edge_index,edge_attr = data.x_left,data.x_right, data.edge_index, data.edge_attr

        left_embed = self.dropout(self.norm0(self.left_embedding(x_left)))
        right_embed = self.dropout(self.norm0(self.right_embedding(x_right)))

        x = torch.cat([left_embed, right_embed], dim=0)
        x1 = self.dropout(torch.relu(self.norm1(self.conv1(x, edge_index,edge_attr=edge_attr))))
        x2 = self.dropout(torch.relu(self.norm2(self.conv2(x1, edge_index,edge_attr=edge_attr))))
        x3 = self.dropout(torch.relu(self.norm3(self.conv3(x2, edge_index))))
        
        # For link prediction, concatenate features from both nodes in a possible link
        start, end = edge_index
        link_repr = torch.cat([x3[start], x3[end]], dim=1)
        
        link_feat = self.dropout(torch.relu(self.norm4(self.link_features(link_repr))))
        link_feat2 = self.dropout(torch.relu(self.norm5(self.link_features2(link_feat))))
        link_feat3 = self.link_features3(link_feat2)
        
        return link_feat3, x3


In [6]:


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LinkPredictor(2,len(act_matrix1[0]),len(train_edge_features_full[0])).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

data = data.to(device)

def train():
    model.train()
    optimizer.zero_grad()
    link_features, _ = model(data)
    loss = torch.nn.functional.mse_loss(link_features.view(-1).to(device),train_edge_features.view(-1).to(device))
    loss.backward()
    optimizer.step()
    return loss.item()

def validate():
    model.eval()
    v_features,_ = model(Data(x_left=data.x_left,x_right=data.x_right, edge_index=validate_edges,edge_attr=None).to(device))
    v_loss = torch.nn.functional.mse_loss(v_features.view(-1).to(device),validate_edge_features.view(-1).to(device))
    return v_loss

def test():
    model.eval()
    t_features,_ = model(Data(x_left=data.x_left,x_right=data.x_right, edge_index=test_edges,edge_attr=None).to(device))
    t_loss = torch.nn.functional.mse_loss(t_features.view(-1).to(device),test_edge_features.view(-1).to(device))
    return t_loss

for epoch in range(100):
    loss = train()
    v_loss = validate()
    print(f"Epoch: {epoch+1}, Train Loss: {loss:.4f}")
    print(f"Epoch: {epoch+1}, Validate Loss: {v_loss:.4f}")

t_lost = test()
print(f"Test Loss: {t_lost:.4f}")

Epoch: 1, Train Loss: 6.2543
Epoch: 1, Validate Loss: 3.0499
Epoch: 2, Train Loss: 3.5541
Epoch: 2, Validate Loss: 2.1616
Epoch: 3, Train Loss: 2.6306
Epoch: 3, Validate Loss: 1.6415
Epoch: 4, Train Loss: 2.0737
Epoch: 4, Validate Loss: 1.2836
Epoch: 5, Train Loss: 1.6745
Epoch: 5, Validate Loss: 1.0147
Epoch: 6, Train Loss: 1.3844
Epoch: 6, Validate Loss: 0.8059
Epoch: 7, Train Loss: 1.1479
Epoch: 7, Validate Loss: 0.6401
Epoch: 8, Train Loss: 0.9489
Epoch: 8, Validate Loss: 0.5060
Epoch: 9, Train Loss: 0.7802
Epoch: 9, Validate Loss: 0.3970
Epoch: 10, Train Loss: 0.6425
Epoch: 10, Validate Loss: 0.3101
Epoch: 11, Train Loss: 0.5262
Epoch: 11, Validate Loss: 0.2436
Epoch: 12, Train Loss: 0.4479
Epoch: 12, Validate Loss: 0.1934
Epoch: 13, Train Loss: 0.3811
Epoch: 13, Validate Loss: 0.1580
Epoch: 14, Train Loss: 0.3277
Epoch: 14, Validate Loss: 0.1351
Epoch: 15, Train Loss: 0.3006
Epoch: 15, Validate Loss: 0.1209
Epoch: 16, Train Loss: 0.2801
Epoch: 16, Validate Loss: 0.1111
Epoch: 17,

In [7]:
import pandas as pd
model.eval()
t_features,_ = model(Data(x_left=data.x_left,x_right=data.x_right, edge_index=test_edges).to(device))
pred_features = [x[0] for x in t_features.cpu().detach().numpy()]
test_features = [x[0] for x in test_edge_features.cpu().detach().numpy()]
submission = pd.DataFrame({
    'rating': [x[0] for x in test_edge_features.cpu().detach().numpy()],
    'pred_rating': [x[0] for x in t_features.cpu().detach().numpy()]
})
utils.printErrorParametersForModelData(submission)

Mean Squared Error (MSE): 0.20748797059059143
Mean Absolute Error (MAE): 0.14530275762081146
R Squared score (R2): 0.3460355408627509
